In [36]:
import pandas as pd
import datetime
import numpy as np
import matplotlib.pyplot as plt

import ee
print(ee.__version__)
ee.Authenticate()  # Regular auth — follow the link and paste the token
ee.Initialize(project='utility-realm-255220')

1.5.21


In [37]:
import geemap
geemap.ee_initialize()

In [38]:
import geemap
import ee

def plot_shooting_with_bbox(lat, lon, bbox_geom, zoom=15):
    """
    Plot a shooting location and an associated bounding box on an interactive map.

    Args:
        lat (float): Latitude of the shooting location
        lon (float): Longitude of the shooting location
        bbox_geom (ee.Geometry.BBox or ee.Geometry.Rectangle): The bounding box geometry for analysis
        zoom (int): Zoom level for the map
    """
    # Initialize Earth Engine if not done already
    # try:
    #     ee.Initialize()
    # except:
    #     ee.Authenticate()
    #     ee.Initialize()

    # Create shooting location point
    point = ee.Geometry.Point([lon, lat])
    point_fc = ee.FeatureCollection([ee.Feature(point)])
    bbox_fc = ee.FeatureCollection([ee.Feature(bbox_geom)])

    # Create and display map
    Map = geemap.Map(center=[lat, lon], zoom=zoom)
    Map.addLayer(point_fc.style(color='red', pointSize=6), {}, 'Shooting Location')
    Map.addLayer(bbox_fc.style(color='blue', fillColor='00000000', width=2), {}, 'Bounding Box')
    Map.addLayerControl()
    return Map



In [8]:
# Lewiston 2023 coordinates and bbox
lat = 44.095108
lon = -70.215545
lewiston_bbox = ee.Geometry.BBox(-70.2600, 44.0600, -70.1700, 44.1300)

# Plot
plot_shooting_with_bbox(lat, lon, lewiston_bbox)


Map(center=[44.095108, -70.215545], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sea…

In [20]:
# Lewiston 2023 coordinates and bbox

lat = 33.132682
lon = -96.662071
allen_bbox = ee.Geometry.BBox(-96.7140, 33.0700, -96.6240, 33.1400)

plot_shooting_with_bbox(lat, lon, allen_bbox)

Map(center=[33.132682, -96.662071], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sea…

In [35]:
lat = 29.199261
lon = -99.787959

uvalde_bbox = ee.Geometry.BBox(-99.807959, 29.179261, -99.767959, 29.219261)

plot_shooting_with_bbox(lat, lon, uvalde_bbox)

Map(center=[29.199261, -99.787959], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Sea…

In [39]:
def process_nlcd_landcover_trends(roi):
    import pandas as pd

    # NLCD classes and groups
    nlcd_classes = {
        11: 'Open Water',
        21: 'Developed, Open Space',
        22: 'Developed, Low Intensity',
        23: 'Developed, Medium Intensity',
        24: 'Developed, High Intensity',
        31: 'Barren Land',
        41: 'Deciduous Forest',
        42: 'Evergreen Forest',
        43: 'Mixed Forest',
        51: 'Dwarf Scrub',
        52: 'Shrub/Scrub',
        71: 'Grassland/Herbaceous',
        81: 'Pasture/Hay',
        82: 'Cultivated Crops',
        90: 'Woody Wetlands',
        95: 'Emergent Herbaceous Wetlands'
    }

    nlcd_groups = {
        'Open Water': 'Water',
        'Developed, Open Space': 'Developed',
        'Developed, Low Intensity': 'Developed',
        'Developed, Medium Intensity': 'Developed',
        'Developed, High Intensity': 'Developed', 'Barren Land': 'Barren',
        'Deciduous Forest': 'Forest', 'Evergreen Forest': 'Forest', 'Mixed Forest': 'Forest',
        'Dwarf Scrub': 'Shrubland', 'Shrub/Scrub': 'Shrubland',
        'Grassland/Herbaceous': 'Grassland', 'Pasture/Hay': 'Agriculture',
        'Cultivated Crops': 'Agriculture', 'Woody Wetlands': 'Wetlands',
        'Emergent Herbaceous Wetlands': 'Wetlands'
    }

    group_order = [
        'Water', 'Developed', 'Forest', 'Shrubland',
        'Grassland', 'Agriculture', 'Wetlands', 'Barren'
    ]

    # Years for 2019 release
    nlcd_years = [2001, 2004, 2006, 2008, 2011, 2013, 2016, 2019]
    results = []

    # Process older NLCD years
    for year in nlcd_years:
        image = ee.Image(f'USGS/NLCD_RELEASES/2019_REL/NLCD/{year}') \
            .select('landcover').clip(roi)
        stats = image.reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=roi,
            scale=30,
            maxPixels=1e9
        )
        freq_dict = stats.get('landcover').getInfo()
        total = sum(freq_dict.values())
        for class_id, label in nlcd_classes.items():
            count = freq_dict.get(str(class_id), 0)
            percent = round(100 * count / total, 2)
            results.append({'Year': year, 'Land Cover': label, 'Percent': percent})

    df_2001_2019 = pd.DataFrame(results)
    df_wide = df_2001_2019.pivot(index='Land Cover', columns='Year', values='Percent').fillna(0)

    # Process 2021 NLCD
    image_2021 = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021") \
        .select('landcover').clip(roi)
    stats_2021 = image_2021.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    )
    freq_dict_2021 = stats_2021.get('landcover').getInfo()
    total_2021 = sum(freq_dict_2021.values())

    results_2021 = []
    for class_id, label in nlcd_classes.items():
        count = freq_dict_2021.get(str(class_id), 0)
        percent = round(100 * count / total_2021, 2)
        results_2021.append({'Land Cover': label, 2021: percent})

    df_2021 = pd.DataFrame(results_2021)

    # Merge all years
    df_merged = pd.merge(df_wide.reset_index(), df_2021, on='Land Cover', how='outer')

    # Add NLCD Group
    df_merged['NLCD Group'] = df_merged['Land Cover'].map(nlcd_groups)

    # Sort based on group and name
    df_sorted = df_merged.copy()
    # df_sorted['Group Order'] = df_sorted['NLCD Group'].map(lambda x: group_order.index(x))
    # df_sorted = df_sorted.sort_values(by=['Group Order', 'Land Cover']).drop(columns='Group Order')
    # Assign group order
    df_sorted['Group Order'] = df_sorted['NLCD Group'].map(lambda x: group_order.index(x))

    # Define custom land cover order (especially for Developed)
    landcover_order = [
        'Developed, Open Space',
        'Developed, Low Intensity',
        'Developed, Medium Intensity',
        'Developed, High Intensity'
    ]

    # Assign custom order within group
    df_sorted['Land Cover Order'] = df_sorted['Land Cover'].apply(
        lambda x: landcover_order.index(x) if x in landcover_order else 99 + hash(x) % 1000
    )

    # Sort by group and custom order, then clean up
    df_sorted = df_sorted.sort_values(by=['Group Order', 'Land Cover Order']).drop(columns=['Group Order', 'Land Cover Order'])


    # Reorder columns
    cols = ['NLCD Group', 'Land Cover'] + [col for col in df_sorted.columns if col not in ['NLCD Group', 'Land Cover']]
    return df_sorted[cols].reset_index(drop=True)


In [17]:
# For Allen, TX
allen_roi = ee.Geometry.Rectangle([-96.7140, 33.0700, -96.6240, 33.1400])
df_allen = process_nlcd_landcover_trends(allen_roi)
display(df_allen)


,NLCD Group,Land Cover,2001,2004,2006,2008,2011,2013,2016,2019,2021
0,Water,Open Water,0.12,0.10,0.11,0.15,0.12,0.12,0.13,0.11,0.11
1,Developed,"Developed, Open Space",7.87,7.94,8.03,8.39,8.61,8.30,8.20,7.76,7.62
2,Developed,"Developed, Low Intensity",14.24,14.66,14.70,16.15,16.07,16.19,16.57,16.52,16.79
3,Developed,"Developed, Medium Intensity",33.61,35.33,37.70,41.17,42.77,44.06,45.75,47.26,48.13
4,Developed,"Developed, High Intensity",8.32,9.05,9.89,11.50,12.38,12.64,13.65,14.95,15.27
5,Forest,Mixed Forest,0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.02
6,Forest,Deciduous Forest,5.66,5.28,4.93,4.65,4.45,4.31,4.14,3.95,3.89
7,Forest,Evergreen Forest,0.14,0.10,0.09,0.09,0.09,0.09,0.09,0.07,0.07
8,Shrubland,Dwarf Scrub,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,Shrubland,Shrub/Scrub,0.15,0.11,0.13,0.16,0.11,0.14,0.14,0.10,0.10


In [32]:
uvalde_bbox = ee.Geometry.BBox(-99.807959, 29.179261, -99.767959, 29.219261)
df_uvalde = process_nlcd_landcover_trends(uvalde_bbox)
display(df_uvalde)

,NLCD Group,Land Cover,2001,2004,2006,2008,2011,2013,2016,2019,2021
0,Water,Open Water,0.49,0.53,0.64,0.76,0.76,0.70,0.71,0.71,0.71
1,Developed,"Developed, Open Space",17.65,16.67,16.37,15.76,15.46,14.99,14.70,14.44,14.39
2,Developed,"Developed, Low Intensity",15.41,15.24,15.24,15.21,15.08,14.94,14.99,14.92,14.95
3,Developed,"Developed, Medium Intensity",12.80,13.74,14.09,14.56,14.98,15.47,15.73,15.97,16.09
4,Developed,"Developed, High Intensity",6.32,6.53,6.57,6.75,6.81,6.93,6.99,7.07,7.07
5,Forest,Evergreen Forest,0.49,0.49,0.49,0.51,0.51,0.51,0.51,0.51,1.63
6,Forest,Deciduous Forest,0.67,0.67,0.69,0.69,0.69,0.69,0.69,0.83,0.83
7,Forest,Mixed Forest,0.38,0.37,0.37,0.37,0.37,0.37,0.37,1.31,1.31
8,Shrubland,Dwarf Scrub,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,Shrubland,Shrub/Scrub,20.40,19.69,19.59,19.19,19.33,19.27,19.05,17.35,16.45


In [15]:
# For Lewiston, ME
lewiston_roi = ee.Geometry.BBox(-70.2600, 44.0600, -70.1700, 44.1300)
df_lewiston = process_nlcd_landcover_trends(lewiston_roi)
display(df_lewiston)

,NLCD Group,Land Cover,2001,2004,2006,2008,2011,2013,2016,2019,2021
0,Water,Open Water,3.83,3.84,3.84,3.83,3.78,3.79,3.77,3.77,3.60
1,Developed,"Developed, High Intensity",11.87,12.04,12.32,12.43,12.52,12.64,12.79,12.92,12.93
2,Developed,"Developed, Low Intensity",15.56,15.65,15.77,15.76,15.83,15.78,15.72,15.59,15.64
3,Developed,"Developed, Medium Intensity",18.27,18.54,18.79,18.96,19.10,19.20,19.42,19.80,19.93
4,Developed,"Developed, Open Space",13.23,13.05,13.69,13.59,13.59,13.52,13.33,12.96,12.92
5,Forest,Deciduous Forest,8.66,8.58,8.41,8.35,8.30,8.29,8.24,8.10,8.06
6,Forest,Evergreen Forest,3.10,3.09,2.99,3.00,2.83,2.83,2.82,2.81,2.68
7,Forest,Mixed Forest,15.00,14.87,14.40,14.27,13.98,14.02,13.84,13.81,13.61
8,Shrubland,Dwarf Scrub,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,Shrubland,Shrub/Scrub,0.10,0.15,0.12,0.16,0.20,0.19,0.12,0.33,0.31


In [22]:
monterey_bbox = ee.Geometry.BBox(
    -118.1684,  # min lon = -118.123869 - 0.045
    34.0276,    # min lat = 34.062567 - 0.035
    -118.0794,  # max lon = -118.123869 + 0.045
    34.0976     # max lat = 34.062567 + 0.035
)

df_monterey_park = process_nlcd_landcover_trends(monterey_bbox )
display(df_monterey_park)

,NLCD Group,Land Cover,2001,2004,2006,2008,2011,2013,2016,2019,2021
0,Water,Open Water,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01,0.01
1,Developed,"Developed, Open Space",5.57,5.47,5.54,5.45,5.40,5.36,5.32,5.17,5.00
2,Developed,"Developed, Low Intensity",12.32,12.25,12.20,12.04,11.98,11.94,11.89,11.75,11.61
3,Developed,"Developed, Medium Intensity",62.62,62.70,62.77,62.92,62.95,62.97,63.00,63.14,63.31
4,Developed,"Developed, High Intensity",16.80,16.90,17.00,17.19,17.28,17.33,17.39,17.56,17.68
5,Forest,Mixed Forest,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07,0.07
6,Forest,Deciduous Forest,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,Forest,Evergreen Forest,0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.02,0.02
8,Shrubland,Dwarf Scrub,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,Shrubland,Shrub/Scrub,1.54,1.54,1.54,1.54,1.53,1.53,1.53,1.53,1.48


# Read Mass Shooting File

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
import pandas as pd

# Set display options
pd.set_option('display.max_rows', None)      # Show all rows
pd.set_option('display.max_columns', None)   # Show all columns
pd.set_option('display.width', None)         # Don't wrap lines
pd.set_option('display.max_colwidth', None)  # Show full content in each cell


In [2]:
!ls /content/drive/MyDrive/urbanization_study/

'2025 The Violence Project Mass Shooter Database - Version 9 (4.8.25) - Full Database.csv'
'2025 The Violence Project Mass Shooter Database - Version 9 (4.8.25) - Full Database.numbers'


In [42]:
df_ms=pd.read_csv('/content/drive/MyDrive/urbanization_study/2025 The Violence Project Mass Shooter Database - Version 9 (4.8.25) - Full Database.csv')

In [5]:
df_ms.shape

(201, 176)

In [43]:
df_ms.head(2)

,Unnamed: 0,Shooter Last Name,Shooter First Name,Full Date,Day of Week,Day,Month,Year,Arrival Time,Arrival to Start Time\n(in Minutes),Start Time,Start to End Time\n(in Minutes),End Time,Time of Day,Street Number,Street Name,City,State,County,Zip Code,Latitude,Longitude,State Code,Region,Urban/Suburban/Rural,Metro/Micro Statistical Area Type,Location,Location Specified,Insider or Outsider,Access Required,Accessed Space,Victims Inside / Outside,Workplace Shooting,Multiple Locations,Other Location Specified,Armed Person on Scene,Specify Armed Person,Number Killed,Total Injured,Shooting Injuries,Other Injuries,Family Member Victim,Romantic Partner Victim,Kidnapping or Hostage Situation,Age,Gender,Race,Height,Weight,Immigrant,Sexual Orientation,Religion,Education,School Performance,School Performance Specified,Birth Order,Number of Siblings,Older Siblings,Younger Siblings,Relationship Status,Children,Employment Status,Employment Type,Military Service,Military Branch,Community Involvement,Community Involvement Specified,Known to Police or FBI,Criminal Record,Part I Crimes,Part II Crimes,Highest Level of Justice System Involvement,History of Physical Altercations,History of Animal Abuse,History of Domestic Abuse,Domestic Abuse Specified,History of Sexual Offenses,Gang Affiliation,Terror Group Affiliation,Known Hate Group or Chat Room Affiliation,Violent Video Games,Bully,Bullied,Raised by Single Parent,Parental Divorce / Separation,Parental Death in Childhood,Parental Suicide,Childhood Trauma,Physically Abused,Sexually Abused,Emotionally Abused,Neglected,Childhood SES,Mother Violent Treatment,Parental Substance Abuse,Parent Criminal Record,Family Member Incarcerated,Adult Trauma,Recent or Ongoing Stressor,Signs of Being in Crisis,Timeline of Signs of Crisis,Crisis Six Months or Less,Signs of Crisis Expanded,Inability to Perform Daily Tasks,Notably Depressed Mood,Unusually Calm or Happy,Rapid Mood Swings,Increased Agitation,Abusive Behavior,Isolation,Losing Touch with Reality,Paranoia,Previous System Contact,System Contact Law Enforcement,System Contact Mental Health,"System Contact ""Other""",Timeframe of First Contact,Timeframe of Most Recent Contact,Suicidality,Prior Hospitalization,Voluntary or Involuntary Hospitalization,Prior Counseling,Voluntary or Mandatory Counseling,Psychiatric Medication,Psychiatric Medication Specified,Medication Category,Treatment 6 Months Prior to Shooting,Mental Illness,FASD (Fetal Alcohol Spectrum Disorder),Known Family Mental Health History,Autism Spectrum,Substance Use,Health Issues,Health Issues - Specify,Head Injury / Possible TBI,Known Prejudices,Motive: Racism/Xenophobia,Motive: Religious Hate,Motive: Misogyny,Motive: Homophobia,Motive: Employment Issue,Motive: Economic Issue,Motive: Legal Issue,Motive: Relationship Issue,Motive: Interpersonal Conflict,Motive: Fame-Seeking,Motive: Other,Motive: Unknown,Role of Psychosis in the Shooting,Social Media Use,Leakage,Leakage How,Leakage Who,Leakage Specific/Nonspecific,Interest in Past Mass Violence,Relationship with Other Shooting(s),Specify Relationship to Other Shooting(s),Legacy Token,Pop Culture Connection,Specify Pop Culture Connection,Planning,Performance,First NYT Page Number,Lowest NYT Page Number,Interest in Firearms,Firearm Proficiency,Total Firearms Brought to the Scene,Other Weapons or Gear,Specify Other Weapons or Gear,On-Scene Outcome,Final Outcome,Attempt to Flee,Apprehended,Who Killed Shooter On Scene,Insanity Defense,Criminal Sentence
0,Case #,Perpetrator Name,NaN,Date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Location,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Victims,NaN,NaN,NaN,NaN,NaN,NaN,Offender Background,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Crime and Violence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Trauma and Adverse Childhood Experiences,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Signs of a Crisis,NaN

In [44]:
df_ms2 = df_ms.iloc[1:].reset_index(drop=True)
df_ms2.shape

(200, 176)

In [19]:
df_ms2.head(1)

,Unnamed: 0,Shooter Last Name,Shooter First Name,Full Date,Day of Week,Day,Month,Year,Arrival Time,Arrival to Start Time\n(in Minutes),Start Time,Start to End Time\n(in Minutes),End Time,Time of Day,Street Number,Street Name,City,State,County,Zip Code,Latitude,Longitude,State Code,Region,Urban/Suburban/Rural,Metro/Micro Statistical Area Type,Location,Location Specified,Insider or Outsider,Access Required,Accessed Space,Victims Inside / Outside,Workplace Shooting,Multiple Locations,Other Location Specified,Armed Person on Scene,Specify Armed Person,Number Killed,Total Injured,Shooting Injuries,Other Injuries,Family Member Victim,Romantic Partner Victim,Kidnapping or Hostage Situation,Age,Gender,Race,Height,Weight,Immigrant,Sexual Orientation,Religion,Education,School Performance,School Performance Specified,Birth Order,Number of Siblings,Older Siblings,Younger Siblings,Relationship Status,Children,Employment Status,Employment Type,Military Service,Military Branch,Community Involvement,Community Involvement Specified,Known to Police or FBI,Criminal Record,Part I Crimes,Part II Crimes,Highest Level of Justice System Involvement,History of Physical Altercations,History of Animal Abuse,History of Domestic Abuse,Domestic Abuse Specified,History of Sexual Offenses,Gang Affiliation,Terror Group Affiliation,Known Hate Group or Chat Room Affiliation,Violent Video Games,Bully,Bullied,Raised by Single Parent,Parental Divorce / Separation,Parental Death in Childhood,Parental Suicide,Childhood Trauma,Physically Abused,Sexually Abused,Emotionally Abused,Neglected,Childhood SES,Mother Violent Treatment,Parental Substance Abuse,Parent Criminal Record,Family Member Incarcerated,Adult Trauma,Recent or Ongoing Stressor,Signs of Being in Crisis,Timeline of Signs of Crisis,Crisis Six Months or Less,Signs of Crisis Expanded,Inability to Perform Daily Tasks,Notably Depressed Mood,Unusually Calm or Happy,Rapid Mood Swings,Increased Agitation,Abusive Behavior,Isolation,Losing Touch with Reality,Paranoia,Previous System Contact,System Contact Law Enforcement,System Contact Mental Health,"System Contact ""Other""",Timeframe of First Contact,Timeframe of Most Recent Contact,Suicidality,Prior Hospitalization,Voluntary or Involuntary Hospitalization,Prior Counseling,Voluntary or Mandatory Counseling,Psychiatric Medication,Psychiatric Medication Specified,Medication Category,Treatment 6 Months Prior to Shooting,Mental Illness,FASD (Fetal Alcohol Spectrum Disorder),Known Family Mental Health History,Autism Spectrum,Substance Use,Health Issues,Health Issues - Specify,Head Injury / Possible TBI,Known Prejudices,Motive: Racism/Xenophobia,Motive: Religious Hate,Motive: Misogyny,Motive: Homophobia,Motive: Employment Issue,Motive: Economic Issue,Motive: Legal Issue,Motive: Relationship Issue,Motive: Interpersonal Conflict,Motive: Fame-Seeking,Motive: Other,Motive: Unknown,Role of Psychosis in the Shooting,Social Media Use,Leakage,Leakage How,Leakage Who,Leakage Specific/Nonspecific,Interest in Past Mass Violence,Relationship with Other Shooting(s),Specify Relationship to Other Shooting(s),Legacy Token,Pop Culture Connection,Specify Pop Culture Connection,Planning,Performance,First NYT Page Number,Lowest NYT Page Number,Interest in Firearms,Firearm Proficiency,Total Firearms Brought to the Scene,Other Weapons or Gear,Specify Other Weapons or Gear,On-Scene Outcome,Final Outcome,Attempt to Flee,Apprehended,Who Killed Shooter On Scene,Insanity Defense,Criminal Sentence
0,1,Whitman,Charles,8/1/1966,Monday,1.0,8.0,1966,11:30 AM,0.0,11:30 AM,114.0,1:24 PM,0.0,110,Inner Campus Dr,Austin,TX,Travis County,78705.0,30.286058,-97.73935,43.0,0.0,0.0,metropolitan,1.0,University of Texas,1.0,2.0,0.0,2.0,0.0,1.0,Shooter's home,1.0,2,15,31.0,28.0,3.0,1.0,1.0,0,25,0.0,0,72.0,200.0,0.0,0.0,1.0,2.0,0.0,"Had a history of good grades in grade school, during his final two years his grades dropped. In college his grades were poor - 1.9 GPA. Dropped out of UT in Feb. 1963. In 1965 he re-enr

In [15]:
df_ms2['Year'].dtypes

dtype('O')

Index(['Unnamed: 0', 'Shooter Last Name', 'Shooter First Name', 'Full Date',
       'Day of Week', 'Day', 'Month', 'Year', 'Arrival Time',
       'Arrival to Start Time\n(in Minutes)',
       ...
       'Total Firearms Brought to the Scene', 'Other Weapons or Gear',
       'Specify Other Weapons or Gear', 'On-Scene Outcome', 'Final Outcome',
       'Attempt to Flee', 'Apprehended', 'Who Killed Shooter On Scene',
       'Insanity Defense', 'Criminal Sentence'],
      dtype='object', length=176)

In [45]:
df_ms2.rename(columns={'Unnamed: 0':'ID'}, inplace=True)

In [46]:
df_ms2['Full Date'] = pd.to_datetime(df_ms2['Full Date'], errors='coerce')
df_ms2['Day'] = pd.to_numeric(df_ms2['Day'], errors='coerce')
df_ms2['Month'] = pd.to_numeric(df_ms2['Month'], errors='coerce')
df_ms2['Year'] = pd.to_numeric(df_ms2['Year'], errors='coerce')

In [24]:
# df_ms2['Year'].value_counts()

In [47]:
df_ms2[df_ms2['Full Date']>='01/01/2013'].shape

(71, 176)

In [49]:
df_2013_2024 = df_ms2[df_ms2['Full Date']>='01/01/2013']
df_2013_2024.shape

(71, 176)

In [52]:
df_2013_2024['RecordID'] = df_2013_2024['City'] + '-'+df_2013_2024['Full Date'].astype(str)

In [53]:
df_2013_2024['RecordID'].head(5)

,RecordID
128,Herkimer-2013-03-13
129,Federal Way -2013-04-21
130,Santa Monica-2013-06-07
131,Hialeah-2013-07-26
132,Washington D.C.-2013-09-16


In [48]:
import math
import ee

def create_square_bbox(lat, lon, side_km=4.4):
    """Return an Earth Engine square bbox (~side_km x side_km) around the lat/lon center."""
    # Convert side length to degree deltas
    delta_lat = (side_km / 2) / 111  # 1° lat ≈ 111 km
    delta_lon = (side_km / 2) / (111 * math.cos(math.radians(lat)))  # adjust for latitude

    return ee.Geometry.Rectangle([
        lon - delta_lon,
        lat - delta_lat,
        lon + delta_lon,
        lat + delta_lat
    ])


In [66]:
def create_bbox_coords(lat, lon, side_km=4.4):
    import math
    delta_lat = (side_km / 2) / 111
    delta_lon = (side_km / 2) / (111 * math.cos(math.radians(lat)))
    return [lon - delta_lon, lat - delta_lat, lon + delta_lon, lat + delta_lat]

# Add as a new column with raw bounding box coordinates
df_2013_2024['BBoxCoords'] = df_2013_2024.apply(
    lambda row: create_bbox_coords(row['Latitude'], row['Longitude']),
    axis=1
)


/tmp/ipython-input-66-3146445215.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2013_2024['BBoxCoords'] = df_2013_2024.apply(


In [54]:
nlcd_outputs = []

for idx, row in df_2013_2024.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    roi = create_square_bbox(lat, lon)
    try:
        nlcd_df = process_nlcd_landcover_trends(roi)
        nlcd_df['RecordID'] = row['RecordID']
        nlcd_outputs.append(nlcd_df)
    except Exception as e:
        print(f"Failed at RecordID {row['RecordID']} (lat: {lat}, lon: {lon}): {e}")

In [55]:
df_nlcd_all = pd.concat(nlcd_outputs, ignore_index=True)

In [56]:
df_nlcd_all.shape

(1136, 12)

In [60]:
df_nlcd_all.head(32)

,NLCD Group,Land Cover,2001,2004,2006,2008,2011,2013,2016,2019,2021,RecordID
0,Water,Open Water,4.63,4.67,4.64,4.70,4.68,4.68,4.73,4.39,4.11,Herkimer-2013-03-13
1,Developed,"Developed, Open Space",9.10,8.93,9.25,9.15,9.18,9.08,9.01,8.92,8.90,Herkimer-2013-03-13
2,Developed,"Developed, Low Intensity",11.51,11.40,11.62,11.48,11.54,11.52,11.48,11.41,11.41,Herkimer-2013-03-13
3,Developed,"Developed, Medium Intensity",13.70,13.92,14.15,14.25,14.40,14.48,14.59,14.69,14.71,Herkimer-2013-03-13
4,Developed,"Developed, High Intensity",5.49,5.57,5.66,5.90,5.95,5.99,6.02,6.07,6.08,Herkimer-2013-03-13
5,Forest,Evergreen Forest,2.46,2.47,2.46,2.45,2.44,2.44,2.42,2.42,2.42,Herkimer-2013-03-13
6,Forest,Deciduous Forest,25.98,25.92,25.79,25.76,25.71,26.01,26.23,26.17,26.17,Herkimer-2013-03-13
7,Forest,Mixed Forest,3.27,3.25,3.23,3.22,3.18,3.18,3.20,3.20,3.20,Herkimer-2013-03-13
8,Shrubland,Dwarf Scrub,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Herkimer-2013-03-13
9,Shrubland,Shrub/Scrub,1.29,1.37,1.54,1.24,1.61,1.31,1.05,1.04,1.04,Herkimer-2013-03-13


In [68]:
df_2013_2024['BBoxCoords'].sample(1)

,BBoxCoords
128,"[-75.01259932058288, 43.00541318018018, -74.95837667941713, 43.04505281981982]"


In [70]:
df_2013_2024.drop(['BoundingBox'], axis=1, inplace=True)

/tmp/ipython-input-70-2219551814.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2013_2024.drop(['BoundingBox'], axis=1, inplace=True)


In [71]:
df_2013_2024.columns.to_list()

['ID',
 'Shooter Last Name',
 'Shooter First Name',
 'Full Date',
 'Day of Week',
 'Day',
 'Month',
 'Year',
 'Arrival Time',
 'Arrival to Start Time\n(in Minutes)',
 'Start Time',
 'Start to End Time\n(in Minutes)',
 'End Time',
 'Time of Day',
 'Street Number',
 'Street Name',
 'City',
 'State',
 'County',
 'Zip Code',
 'Latitude',
 'Longitude',
 'State Code',
 'Region',
 'Urban/Suburban/Rural',
 'Metro/Micro Statistical Area Type',
 'Location',
 'Location Specified',
 'Insider or Outsider',
 'Access Required',
 'Accessed Space',
 'Victims Inside / Outside ',
 'Workplace Shooting',
 'Multiple Locations',
 'Other Location Specified',
 'Armed Person on Scene',
 'Specify Armed Person',
 'Number Killed',
 'Total Injured',
 'Shooting Injuries',
 'Other Injuries',
 'Family Member Victim',
 'Romantic Partner Victim',
 'Kidnapping or Hostage Situation',
 'Age',
 'Gender',
 'Race',
 'Height',
 'Weight',
 'Immigrant',
 'Sexual Orientation',
 'Religion',
 'Education',
 'School Performance',
 'S

In [61]:
df_nlcd_all.to_csv('/content/drive/MyDrive/urbanization_study/df_nlcd_all.csv', index=False)

In [72]:
df_2013_2024.to_csv('/content/drive/MyDrive/urbanization_study/df_2013_2024.csv', index=False)

In [62]:
# df_ms2.columns.to_list()

In [30]:
df_ms2[['Full Date',  'Street Number',
 'Street Name',
 'City',
 'State',
 'County',  'Number Killed',
 'Total Injured',
 'Zip Code','Who Killed Shooter On Scene']].tail(10)

,Full Date,Street Number,Street Name,City,State,County,Number Killed,Total Injured,Zip Code,Who Killed Shooter On Scene
190,2023-01-23,12761,CA-92,Half Moon Bay,CA,San Mateo County,7,1.0,94019.0,0.0
191,2023-03-27,33,Burton Hills Blvd,Nashville,TN,Davidson County,6,0.0,37215.0,2.0
192,2023-04-10,333,E Main St Ste 100,Louisville,KY,Jefferson County,5,8.0,40202.0,2.0
193,2023-05-06,820,W Stacy Rd,Allen,TX,Collin County,8,7.0,75013.0,2.0
194,2023-07-03,1600 block,South Frazier Street,Philadelphia,PA,Philadelphia County,5,4.0,19143.0,0.0
195,2023-07-15,NaN,Dogwood Lakes subdivision,Hampton,GA,Henry County,4,3.0,30228.0,2.0
196,2023-10-25,24,Mollison Way,Lewiston,ME,Androscoggin County,18,13.0,4240.0,0.0
197,2024-06-21,920,West 4th Street,Fordyce,AR,Dallas County,4,11.0,71742.0,0.0
198,2024-09-02,7601,Van Buren St,Forest Park,IL,Cook County,4,0.0,60130.0,0.0
199,2024-09-04,940,Haymon Morris Rd,Winder,GA,Barrow County,4,9.0,30680.0,0.0
